# 09 — ML de clustering: ¿qué perfiles de riesgo tienen los edificios?
Segmentación con **KMeans** de MLlib de los edificios de los 6 distritos bajo el escenario de la Norma E.030 con amplificación del suelo (sin etiqueta: se juzga con el *silhouette* y con la interpretación de cada grupo).
1. **Monitoreo** del champion: silhouette sobre el lote nuevo (cambia cuando cambia el champion de amplificación), cambio en la proporción de edificios por cluster (PSI) y drift de las variables.
2. **Decisión** de recalibrado (si no hay evidencia, el notebook termina).
3. **Entrenamiento:** se prueba k de 3 a 7 y se elige el de mayor silhouette.
4. **Promoción:** el challenger reemplaza al champion si su silhouette mejora por un margen mínimo.
5. **MLflow:** cada k probado queda registrado como un run del experimento.

Los clusters se numeran de menor a mayor demanda relativa a la capacidad (`ln_IM_theta`): el cluster 0 es el de menor riesgo.

## 1. Setup — imports

In [ ]:
# Librerías que usa el notebook.
import json

import mlflow
import numpy as np
import pandas as pd

from pyspark.sql.functions import col, count, avg
from pyspark.sql import functions as F

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

## 2. Catálogo, parámetros y umbrales

In [ ]:
# Catálogo y esquema; forzar_recalibrado llega como parámetro del job.
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

dbutils.widgets.text("forzar_recalibrado", "false")
forzar = dbutils.widgets.get("forzar_recalibrado").lower() == "true"

In [ ]:
# Caso, target y umbrales de monitoreo y promoción.
caso = "clustering"
SEED = 700

UMBRALES = {
    "psi_feature": 0.20,
    "n_features_drift": 2,
    "caida_silhouette": 0.05,
    "psi_clusters": 0.20,
    "mejora_minima_silhouette": 0.02,
}

## 3. Función de drift: PSI
Population Stability Index entre la distribución de referencia (datos del champion) y la del lote nuevo. PSI > 0.20 indica un cambio importante.

In [ ]:
# PSI de cada variable entre la referencia y el lote actual.
def psi(referencia, actual, bins=10):
    ref = pd.Series(referencia).dropna().astype(float)
    act = pd.Series(actual).dropna().astype(float)
    cortes = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(cortes) < 3:
        return 0.0
    cortes[0], cortes[-1] = -np.inf, np.inf
    p = np.histogram(ref, cortes)[0] / len(ref)
    q = np.histogram(act, cortes)[0] / len(act)
    p, q = np.clip(p, 1e-4, None), np.clip(q, 1e-4, None)
    return float(np.sum((q - p) * np.log(q / p)))

## 4. Datos y variables seleccionadas

In [ ]:
df = spark.table("fe_clustering")

seleccion = spark.table("fe_columnas_seleccionadas").filter(col("caso") == caso)
cols_nums = [r["columna"] for r in seleccion.filter(col("tipo") == "num").collect()]

lote_actual = spark.table("fe_columnas_seleccionadas").filter(col("caso") == caso).first()["lote"]
actual = df.filter(col("lote") == lote_actual)

print("Lote actual:", lote_actual, "| edificios:", actual.count())
print("Variables seleccionadas:", cols_nums)


# PSI entre dos distribuciones.
def psi_distribucion(p, q):
    p, q = np.clip(np.array(p, dtype=float), 1e-4, None), np.clip(np.array(q, dtype=float), 1e-4, None)
    return float(np.sum((q - p) * np.log(q / p)))


# Proporción de edificios en cada cluster.
def proporciones(modelo, datos):
    k = modelo.stages[-1].getK()
    conteo = modelo.transform(datos).groupBy("cluster").count().toPandas().set_index("cluster")["count"]
    conteo = conteo.reindex(range(k), fill_value=0)
    return (conteo / conteo.sum()).tolist()

## 5. Champion actual (registro de modelos)

In [ ]:
# Carga el champion vigente y sus metadatos.
registro = spark.table("ml_registro_modelos").filter(col("caso") == caso)
fila_champion = registro.filter(col("promovido")).orderBy(col("version").desc()).first()

if fila_champion is None:
    champion, meta = None, None
else:
    champion = PipelineModel.load(fila_champion["ruta"])
    meta = {"version": fila_champion["version"],
            "cols_num": json.loads(fila_champion["columnas_num"]),
            "cols_cat": json.loads(fila_champion["columnas_cat"]),
            "snapshots": json.loads(fila_champion["snapshots"]),
            "metricas": json.loads(fila_champion["metricas"]),
            "detalle": json.loads(fila_champion["detalle"])}

print("Champion actual:", "v" + str(meta["version"]) if meta else "no existe")

## 6. Monitoreo: degradación de la métrica y drift

In [ ]:
# Mide la degradación del champion y el drift de cada variable.
n_drift, metrica_base, metrica_nueva, degradacion = 0, None, None, False
evaluador = ClusteringEvaluator(featuresCol="features_scaled", predictionCol="cluster", metricName="silhouette")

if champion is not None and lote_actual not in meta["snapshots"]:
    ref = df.filter(col("lote").isin(meta["snapshots"])).select(meta["cols_num"]).toPandas()
    act = actual.select(meta["cols_num"]).toPandas()
    psis = {c: psi(ref[c], act[c]) for c in meta["cols_num"]}
    n_drift = sum(v > UMBRALES["psi_feature"] for v in psis.values())
    print("PSI:", [(c, round(v, 3)) for c, v in sorted(psis.items(), key=lambda kv: -kv[1])])

    metrica_base = meta["metricas"]["Silhouette"]
    metrica_nueva = evaluador.evaluate(champion.transform(actual))
    psi_clusters = psi_distribucion(meta["detalle"]["proporciones"], proporciones(champion, actual))
    degradacion = (metrica_base - metrica_nueva > UMBRALES["caida_silhouette"]) or (psi_clusters > UMBRALES["psi_clusters"])
    print(f"Silhouette base: {metrica_base:.3f} | hoy: {metrica_nueva:.3f} | PSI de clusters: {psi_clusters:.3f} | degradacion: {degradacion}")
elif champion is not None:
    print(f"El champion ya se entrenó con {lote_actual}: no hay datos nuevos que monitorear")

## 7. Decisión de recalibrado
Se recalibra si no hay champion, si la métrica se degradó, si hay drift en 2 o más variables o si el job lo fuerza. La decisión queda en `ml_monitoreo`.

In [ ]:
# Decide si recalibrar: sin champion, degradación, drift o pedido del job.
if champion is None:
    reentrenar, motivo = True, "No existe champion: primer entrenamiento"
else:
    drift = n_drift >= UMBRALES["n_features_drift"]
    reentrenar = bool(degradacion or drift or forzar)
    senales = [("degradacion de la metrica", degradacion), (f"drift en {n_drift} variables", drift), ("forzado por parametro", forzar)]
    motivos = [texto for texto, activa in senales if activa]
    motivo = " + ".join(motivos) if motivos else "Sin cambios relevantes: se conserva el champion"

print("DECISION:", "RECALIBRAR" if reentrenar else "SIN CAMBIOS", "->", motivo)

In [ ]:
# Guarda la decisión en ml_monitoreo.
clave = f"{lote_actual}|{caso}"
fila = [(clave, lote_actual, caso, meta["version"] if meta else None, int(n_drift), metrica_base, metrica_nueva,
         bool(degradacion), bool(reentrenar), motivo)]
esquema_monitoreo = ("clave string, lote string, caso string, champion_version int, n_features_drift int, "
                     "metrica_base double, metrica_nueva double, degradacion boolean, reentrenar boolean, motivo string")
monitoreo = spark.createDataFrame(fila, esquema_monitoreo)

monitoreo.write.mode("overwrite").option("replaceWhere", f"caso = '{caso}' AND lote = '{lote_actual}'").saveAsTable("ml_monitoreo")

In [ ]:
# Termina aquí si no hace falta recalibrar.
if not reentrenar:
    dbutils.notebook.exit("Sin cambios: se conserva el champion")

## 8. Entrenamiento

In [ ]:
# Prepara las variables: vector y escalado.
train = actual
snapshots_train = [lote_actual]

num_assembler = VectorAssembler(inputCols=cols_nums, outputCol="num_features")
scaler = StandardScaler(inputCol="num_features", outputCol="features_scaled", withMean=True, withStd=True)

In [ ]:
# Prueba k de 3 a 7 y mide el silhouette.
silhouettes, modelos_k = {}, {}

for k in range(3, 8):
    kmeans = KMeans(k=k, seed=SEED, featuresCol="features_scaled", predictionCol="cluster")
    modelos_k[k] = Pipeline(stages=[num_assembler, scaler, kmeans]).fit(train)
    silhouettes[k] = evaluador.evaluate(modelos_k[k].transform(train))
    print(f"KMeans k={k}: Silhouette={silhouettes[k]:.4f}")

## 9. Perfil de cada cluster

In [ ]:
# Elige el k con mejor silhouette.
k_final = max(silhouettes, key=silhouettes.get)
challenger = modelos_k[k_final]
sil_challenger = silhouettes[k_final]
print("Mejor k:", k_final, "| silhouette:", round(sil_challenger, 4))

medias = challenger.transform(train).groupBy("cluster").agg(avg("ln_IM_theta").alias("m")).orderBy("m").toPandas()
orden = {int(c): i for i, c in enumerate(medias.cluster)}
mapa = F.create_map(*[F.lit(x) for par in orden.items() for x in par])

perfil = (challenger.transform(train).withColumn("cluster", mapa[col("cluster")]).groupBy("cluster")
          .agg(count("*").alias("n"), *[avg(c).alias(c) for c in cols_nums],
               avg("P_colapso").alias("P_colapso"), avg("dano_esperado").alias("dano_esperado"),
               *[(100 * avg((col("tipologia") == t).cast("double"))).alias(f"pct_{t}") for t in ["adobe", "albanileria_confinada", "concreto_armado"]])
          .orderBy("cluster").toPandas())
display(perfil.round(3))

## 10. Promoción champion / challenger

In [ ]:
# Promueve si el silhouette mejora lo suficiente.
if champion is None:
    promover = True
else:
    sil_champion = evaluador.evaluate(champion.transform(train))
    promover = sil_challenger >= sil_champion + UMBRALES["mejora_minima_silhouette"]
    print(f"Silhouette challenger: {sil_challenger:.4f} | champion: {sil_champion:.4f}")

print("Promover challenger a champion:", promover)

## 11. Guardar el modelo en el Volume y registrarlo

In [ ]:
# Métricas y detalle que se guardan con el modelo.
ganador = "KMeans"
metricas = {"Silhouette": float(sil_challenger)}
detalle = {"k": k_final, "proporciones": proporciones(challenger, train), "orden": {str(c): i for c, i in orden.items()},
           "silhouette_por_k": {str(k): round(v, 4) for k, v in silhouettes.items()}, "perfil": perfil.round(4).to_dict("records")}

In [ ]:
# Guarda el modelo en el Volume y lo registra.
version = registro.count() + 1
ruta = f"{volumen}/modelos/{caso}/v{version}"
challenger.write().overwrite().save(ruta)

fila = [(caso, version, ganador, lote_actual, ruta, bool(promover), json.dumps(cols_nums), json.dumps([]),
         json.dumps(snapshots_train), json.dumps(metricas), json.dumps(detalle, default=float))]
esquema_registro = ("caso string, version int, algoritmo string, lote string, ruta string, promovido boolean, "
                    "columnas_num string, columnas_cat string, snapshots string, metricas string, detalle string")
spark.createDataFrame(fila, esquema_registro).write.mode("append").saveAsTable("ml_registro_modelos")

print(f"Version v{version} guardada en {ruta}")
print("Estado:", "CHAMPION" if promover else "challenger (se conserva el champion actual)")

## 12. MLflow — experimento
Cada ejecución del notebook queda registrada en un experimento de MLflow (un *run* padre por ejecución y un *run* hijo por algoritmo) para comparar parámetros y métricas desde la interfaz.

In [ ]:
# Experimento de MLflow del caso.
usuario = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{usuario}/sismos_{caso}")

In [ ]:
# Registra parámetros, métricas y modelo en MLflow.
with mlflow.start_run(run_name=f"{caso}_{lote_actual}"):
    mlflow.log_params({"lote": lote_actual, "forzado": forzar, "n_edificios": train.count(), "variables": ",".join(cols_nums),
                       "k_final": k_final, "promovido": bool(promover), "version_registro": version, "ruta_modelo": ruta})
    mlflow.log_metric("silhouette_challenger", sil_challenger)
    if champion is not None:
        mlflow.log_metric("silhouette_champion", sil_champion)

    for k, silueta in silhouettes.items():
        with mlflow.start_run(run_name=f"KMeans_k{k}", nested=True):
            mlflow.log_param("k", k)
            mlflow.log_metric("silhouette", float(silueta))